In [ ]:


import argparse
import math
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

NUM_CLASSES = 19
IMAGE_SIZE = (512, 1024)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CRF_ITERATIONS = 5

CITYSCAPES_COLORS = np.array([
    [128,  64, 128], [244,  35, 232], [ 70,  70,  70], [102, 102, 156],
    [190, 153, 153], [153, 153, 153], [250, 170,  30], [220, 220,   0],
    [107, 142,  35], [152, 251, 152], [ 70, 130, 180], [220,  20,  60],
    [255,   0,   0], [  0,   0, 142], [  0,   0,  70], [  0,  60, 100],
    [  0,  80, 100], [  0,   0, 230], [119,  11,  32],
], dtype=np.uint8)

CLASS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence",
    "pole", "traffic light", "traffic sign", "vegetation", "terrain",
    "sky", "person", "rider", "car", "truck",
    "bus", "train", "motorcycle", "bicycle",
]




class ResNet50Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        from torchvision.models import resnet50, ResNet50_Weights
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool)
        self.layer1 = m.layer1
        self.layer2 = m.layer2
        self.layer3 = m.layer3
        self.layer4 = m.layer4
        for module in [self.stem, self.layer1, self.layer2]:
            for p in module.parameters():
                p.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        self.stem.eval()
        self.layer1.eval()
        self.layer2.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            x = self.stem(x)
            c1 = self.layer1(x)
            c2 = self.layer2(c1)
        c3 = self.layer3(c2.detach())
        c4 = self.layer4(c3)
        return c1, c2, c3, c4


class EdgeBranch(nn.Module):
    def __init__(self, out_channels=(64, 128, 256, 512)):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3))
        self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3))
        c1, c2, c3, c4 = out_channels
        self.stem = nn.Sequential(
            nn.Conv2d(5, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, c1, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.Conv2d(c1, c1, 3, padding=1, bias=False),
            nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
        )
        self.block2 = self._block(c1, c2)
        self.block3 = self._block(c2, c3)
        self.block4 = self._block(c3, c4)
        self.edge_head = nn.Conv2d(c1, 1, 1)

    @staticmethod
    def _block(in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )

    def _sobel(self, x):
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        sx = F.conv2d(gray, self.sobel_x, padding=1)
        sy = F.conv2d(gray, self.sobel_y, padding=1)
        return torch.cat([sx, sy], dim=1)

    def forward(self, x):
        z = torch.cat([x, self._sobel(x)], dim=1)
        e1 = self.stem(z)
        e2 = self.block2(e1)
        e3 = self.block3(e2)
        e4 = self.block4(e3)
        edge_logit = self.edge_head(e1)
        return (e1, e2, e3, e4), edge_logit


class CrossAttnFusion(nn.Module):
    def __init__(self, dim_a, dim_b, out_dim, num_heads=4):
        super().__init__()
        self.proj_a = nn.Conv2d(dim_a, out_dim, 1)
        self.proj_b = nn.Conv2d(dim_b, out_dim, 1)
        self.norm_a = nn.LayerNorm(out_dim)
        self.norm_b = nn.LayerNorm(out_dim)
        self.attn_a2b = nn.MultiheadAttention(out_dim, num_heads, batch_first=True)
        self.attn_b2a = nn.MultiheadAttention(out_dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim * 2), nn.GELU(),
            nn.Linear(out_dim * 2, out_dim),
        )
        self.norm_out = nn.LayerNorm(out_dim)

    def forward(self, a, b):
        B, _, H, W = a.shape
        a = self.proj_a(a)
        b = self.proj_b(b)
        C = a.size(1)
        a_seq = a.flatten(2).transpose(1, 2)
        b_seq = b.flatten(2).transpose(1, 2)
        a_seq = self.norm_a(a_seq)
        b_seq = self.norm_b(b_seq)
        a_att, _ = self.attn_a2b(a_seq, b_seq, b_seq)
        b_att, _ = self.attn_b2a(b_seq, a_seq, a_seq)
        fused = torch.cat([a_seq + a_att, b_seq + b_att], dim=-1)
        fused = self.ffn(fused) + a_seq
        fused = self.norm_out(fused)
        return fused.transpose(1, 2).reshape(B, C, H, W)


class LightFusion(nn.Module):
    def __init__(self, dim_a, dim_b, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(dim_a + dim_b, out_dim, 1, bias=False),
            nn.BatchNorm2d(out_dim), nn.ReLU(inplace=True),
        )

    def forward(self, a, b):
        return self.proj(torch.cat([a, b], dim=1))


class CRFasRNN(nn.Module):
    def __init__(self, num_classes, num_iter=5, ksize=5):
        super().__init__()
        self.num_classes = num_classes
        self.num_iter = num_iter
        self.ksize = ksize
        self.smooth = nn.Conv2d(num_classes, num_classes, ksize,
                                padding=ksize // 2, groups=num_classes, bias=False)
        self.appearance = nn.Conv2d(num_classes + 3, num_classes, ksize,
                                    padding=ksize // 2, bias=False)
        self.w_smooth = nn.Parameter(torch.tensor(1.0))
        self.w_appear = nn.Parameter(torch.tensor(1.0))
        self.compat = nn.Conv2d(num_classes, num_classes, 1, bias=False)
        with torch.no_grad():
            g = self._gaussian(ksize, sigma=1.0)
            for c in range(num_classes):
                self.smooth.weight[c, 0].copy_(g)
            nn.init.xavier_uniform_(self.appearance.weight)
            w = torch.eye(num_classes).neg().view(num_classes, num_classes, 1, 1)
            self.compat.weight.copy_(w)

    @staticmethod
    def _gaussian(ksize, sigma):
        ax = torch.arange(ksize, dtype=torch.float32) - ksize // 2
        yy, xx = torch.meshgrid(ax, ax, indexing="ij")
        g = torch.exp(-(xx * xx + yy * yy) / (2 * sigma * sigma))
        return g / g.sum()

    def forward(self, unary, image):
        if image.shape[-2:] != unary.shape[-2:]:
            image = F.interpolate(image, size=unary.shape[-2:],
                                  mode="bilinear", align_corners=False)
        q = F.softmax(unary, dim=1)
        for _ in range(self.num_iter):
            m_smooth = self.smooth(q)
            m_appear = self.appearance(torch.cat([q, image], dim=1))
            message = self.w_smooth * m_smooth + self.w_appear * m_appear
            pairwise = self.compat(message)
            q = F.softmax(unary - pairwise, dim=1)
        return q


class MultiScaleCRFDecoder(nn.Module):
    def __init__(self, fused_channels, num_classes=NUM_CLASSES,
                 decoder_dim=256, crf_iter=CRF_ITERATIONS):
        super().__init__()
        d = decoder_dim
        f1, f2, f3, f4 = fused_channels
        self.lat4 = nn.Conv2d(f4, d, 1)
        self.lat3 = nn.Conv2d(f3, d, 1)
        self.lat2 = nn.Conv2d(f2, d, 1)
        self.lat1 = nn.Conv2d(f1, d, 1)
        self.smooth4 = self._smooth(d)
        self.smooth3 = self._smooth(d)
        self.smooth2 = self._smooth(d)
        self.smooth1 = self._smooth(d)
        self.head_q = nn.Conv2d(d, num_classes, 1)
        self.up_q2h = nn.Sequential(
            nn.Conv2d(d, d // 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(d // 2), nn.ReLU(inplace=True),
        )
        self.head_h = nn.Conv2d(d // 2, num_classes, 1)
        self.up_h2f = nn.Sequential(
            nn.Conv2d(d // 2, d // 4, 3, padding=1, bias=False),
            nn.BatchNorm2d(d // 4), nn.ReLU(inplace=True),
        )
        self.head_f = nn.Conv2d(d // 4, num_classes, 1)
        self.crf_q = CRFasRNN(num_classes, num_iter=crf_iter)
        self.crf_h = CRFasRNN(num_classes, num_iter=crf_iter)
        self.crf_f = CRFasRNN(num_classes, num_iter=crf_iter)

    @staticmethod
    def _smooth(d):
        return nn.Sequential(
            nn.Conv2d(d, d, 3, padding=1, bias=False),
            nn.BatchNorm2d(d), nn.ReLU(inplace=True),
        )

    def forward(self, fused_feats, image):
        f1, f2, f3, f4 = fused_feats
        p4 = self.smooth4(self.lat4(f4))
        p3 = self.smooth3(self.lat3(f3) + F.interpolate(p4, size=f3.shape[-2:],
                                                        mode="bilinear", align_corners=False))
        p2 = self.smooth2(self.lat2(f2) + F.interpolate(p3, size=f2.shape[-2:],
                                                        mode="bilinear", align_corners=False))
        p1 = self.smooth1(self.lat1(f1) + F.interpolate(p2, size=f1.shape[-2:],
                                                        mode="bilinear", align_corners=False))
        unary_q = self.head_q(p1)
        prob_q = self.crf_q(unary_q, image)
        p_h = F.interpolate(p1, scale_factor=2, mode="bilinear", align_corners=False)
        p_h = self.up_q2h(p_h)
        unary_h = self.head_h(p_h)
        prob_h = self.crf_h(unary_h, image)
        p_f = F.interpolate(p_h, scale_factor=2, mode="bilinear", align_corners=False)
        p_f = self.up_h2f(p_f)
        unary_f = self.head_f(p_f)
        prob_f = self.crf_f(unary_f, image)
        return {
            "unary_q": unary_q, "prob_q": prob_q,
            "unary_h": unary_h, "prob_h": prob_h,
            "unary_f": unary_f, "prob_f": prob_f,
        }


class DualPathCRFNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, decoder_dim=256, crf_iter=CRF_ITERATIONS):
        super().__init__()
        self.backbone = ResNet50Backbone()
        self.edge = EdgeBranch(out_channels=(64, 128, 256, 512))
        self.fuse1 = LightFusion(256, 64, 256)
        self.fuse2 = LightFusion(512, 128, 256)
        self.fuse3 = CrossAttnFusion(1024, 256, 256, num_heads=4)
        self.fuse4 = CrossAttnFusion(2048, 512, 256, num_heads=4)
        self.decoder = MultiScaleCRFDecoder(
            fused_channels=(256, 256, 256, 256),
            num_classes=num_classes,
            decoder_dim=decoder_dim,
            crf_iter=crf_iter,
        )

    def forward(self, image):
        c1, c2, c3, c4 = self.backbone(image)
        (e1, e2, e3, e4), edge_logit = self.edge(image)
        f1 = self.fuse1(c1, e1)
        f2 = self.fuse2(c2, e2)
        f3 = self.fuse3(c3, e3)
        f4 = self.fuse4(c4, e4)
        out = self.decoder((f1, f2, f3, f4), image)
        out["edge_logit"] = edge_logit
        return out


# GRAD-CAM ENGINE


class GradCAM:


    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        # Register hooks
        self._fwd_hook = target_layer.register_forward_hook(self._save_activation)
        self._bwd_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx, pred_mask=None):
        """
        Generate Grad-CAM heatmap for a specific class.

        Args:
            input_tensor: (1, 3, H, W) normalized input
            class_idx: target class index (0-18)
            pred_mask: optional precomputed prediction mask; if None, computed here

        Returns:
            heatmap: (H_input, W_input) numpy array in [0, 1]
        """
        self.model.zero_grad()

        # Forward pass (need grads for this)
        out = self.model(input_tensor)
        logits = out["unary_f"]  # use pre-CRF logits for cleaner gradients

        # Get prediction mask if not provided
        if pred_mask is None:
            pred_mask = out["prob_f"].argmax(1).squeeze(0)

        # Create spatial score: sum of logits at pixels predicted as class_idx
        class_mask = (pred_mask == class_idx).float().unsqueeze(0).unsqueeze(0)

        # Resize mask to logits spatial size if needed
        if class_mask.shape[-2:] != logits.shape[-2:]:
            class_mask = F.interpolate(class_mask, size=logits.shape[-2:],
                                       mode="nearest")

        # Score = sum of logit values for target class at predicted locations
        score = (logits[0, class_idx:class_idx+1] * class_mask[0]).sum()

        if score.item() == 0:
            # Class not present in prediction
            h, w = input_tensor.shape[-2:]
            return np.zeros((h, w), dtype=np.float32)

        # Backward
        score.backward(retain_graph=False)

        # Grad-CAM computation
        grads = self.gradients[0]        # (C, h, w)
        acts = self.activations[0]       # (C, h, w)

        # Global average pool gradients -> channel weights
        weights = grads.mean(dim=(1, 2))  # (C,)

        # Weighted combination of activation maps
        cam = (weights[:, None, None] * acts).sum(dim=0)  # (h, w)
        cam = F.relu(cam)  # only positive contributions

        # Normalize to [0, 1]
        cam_max = cam.max()
        if cam_max > 0:
            cam = cam / cam_max

        # Resize to input resolution
        cam = cam.unsqueeze(0).unsqueeze(0)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:],
                            mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        return cam

    def remove_hooks(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()


def load_model(ckpt_path, device):
    model = DualPathCRFNet().to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        state = ckpt.get("model", ckpt)
    else:
        state = ckpt
    model.load_state_dict(state, strict=False)
    # Keep in train mode so BN layers allow gradient flow,
    # but freeze backbone manually
    model.eval()
    return model


def preprocess(image_path, device):
    img = cv2.imread(image_path)
    assert img is not None, f"Cannot read: {image_path}"
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    oh, ow = rgb.shape[:2]

    H, W = IMAGE_SIZE
    resized = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_LINEAR)

    x = torch.from_numpy(resized).permute(2, 0, 1).float() / 255.0
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    x = (x - mean) / std
    x = x.unsqueeze(0).to(device)
    x.requires_grad_(True)

    return rgb, x, (oh, ow)


def apply_colormap(heatmap, colormap=cv2.COLORMAP_JET):
    """Convert [0,1] heatmap to RGB colormap."""
    heatmap_u8 = (heatmap * 255).clip(0, 255).astype(np.uint8)
    colored = cv2.applyColorMap(heatmap_u8, colormap)
    return cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)


def overlay_heatmap(rgb, heatmap, alpha=0.5):
    """Blend RGB image with heatmap."""
    colored = apply_colormap(heatmap)
    blended = alpha * rgb.astype(np.float32) + (1 - alpha) * colored.astype(np.float32)
    return blended.clip(0, 255).astype(np.uint8)


def get_layer_by_name(model, layer_name):
    """Resolve a dot-separated layer name to the actual module."""
    parts = layer_name.split(".")
    module = model
    for p in parts:
        if p.isdigit():
            module = module[int(p)]
        else:
            module = getattr(module, p)
    return module


def main(
    checkpoint=None,
    image=None,
    output="gradcam_output.png",
    classes=None,
    only_present=True,
    target_layer_name="decoder.smooth1",
    alpha=0.5,
    individual=False,
):
    """
    Run Grad-CAM analysis.

    Can be called directly in a notebook:
        main(checkpoint="best.pt", image="input.png")

    Or from command line:
        python gradcam_semseg.py   checkpoint best.pt   image input.png
    """
    #   Handle command-line usage via argparse (skipped in notebooks)  
    if checkpoint is None or image is None:
        parser = argparse.ArgumentParser(
            description="DualPathCRFNet — Grad-CAM Per-Class Visualization",
            formatter_class=argparse.RawTextHelpFormatter,
        )
        parser.add_argument("  checkpoint", required=True, help="Path to best.pt")
        parser.add_argument("  image", required=True, help="Path to input image")
        parser.add_argument("  output", default="gradcam_output.png")
        parser.add_argument("  classes", nargs="*", default=None)
        parser.add_argument("  only-present", action="store_true")
        parser.add_argument("  target-layer", default="decoder.smooth1")
        parser.add_argument("  alpha", type=float, default=0.5)
        parser.add_argument("  individual", action="store_true")
        args = parser.parse_args()
        checkpoint = args.checkpoint
        image = args.image
        output = args.output
        classes = args.classes
        only_present = args.only_present
        target_layer_name = args.target_layer
        alpha = args.alpha
        individual = args.individual

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Load model
    print(f"Loading checkpoint: {checkpoint}")
    model = load_model(checkpoint, device)

    # Resolve target layer
    target_layer = get_layer_by_name(model, target_layer_name)
    print(f"Target layer: {target_layer_name} -> {target_layer.__class__.__name__}")

    # Preprocess image
    print(f"Image: {image}")
    rgb, x, (oh, ow) = preprocess(image, device)
    rgb_resized = cv2.resize(rgb, (IMAGE_SIZE[1], IMAGE_SIZE[0]))

    # Get prediction first (for masking and filtering)
    with torch.no_grad():
        out = model(x)
        pred = out["prob_f"].argmax(1).squeeze(0)
        pred_np = pred.cpu().numpy().astype(np.uint8)

    # Determine which classes to visualize
    if classes:
        name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}
        class_indices = []
        for name in classes:
            if name in name_to_idx:
                class_indices.append(name_to_idx[name])
            elif name.isdigit() and int(name) < NUM_CLASSES:
                class_indices.append(int(name))
            else:
                print(f"[WARN] Unknown class: {name}, skipping")
    elif only_present:
        present = np.unique(pred_np)
        class_indices = [int(c) for c in present if c < NUM_CLASSES]
        print(f"Classes present in prediction: {[CLASS_NAMES[c] for c in class_indices]}")
    else:
        class_indices = list(range(NUM_CLASSES))

    if not class_indices:
        print("No classes to visualize!")
        return

    # Output directory + image stem for unique filenames
    import os
    out_dir = os.path.dirname(os.path.abspath(output))
    os.makedirs(out_dir, exist_ok=True)
    img_stem = os.path.splitext(os.path.basename(image))[0]

    # Save original image separately
    orig_save = os.path.join(out_dir, f"{img_stem}_00_original.png")
    cv2.imwrite(orig_save, cv2.cvtColor(rgb_resized, cv2.COLOR_RGB2BGR))
    print(f"  Saved: {orig_save}")

    # Save prediction mask separately
    pred_resized = cv2.resize(pred_np, (IMAGE_SIZE[1], IMAGE_SIZE[0]),
                              interpolation=cv2.INTER_NEAREST)
    pred_colored = CITYSCAPES_COLORS[pred_resized]
    pred_save = os.path.join(out_dir, f"{img_stem}_01_prediction_mask.png")
    cv2.imwrite(pred_save, cv2.cvtColor(pred_colored, cv2.COLOR_RGB2BGR))
    print(f"  Saved: {pred_save}")

    # Generate Grad-CAM for all present classes
    gradcam = GradCAM(model, target_layer)
    heatmaps = {}

    print(f"\nGenerating Grad-CAM for {len(class_indices)} classes...")
    for cls_idx in class_indices:
        name = CLASS_NAMES[cls_idx]
        x_fresh = x.detach().clone().requires_grad_(True)
        cam = gradcam.generate(x_fresh, cls_idx, pred_mask=pred)
        pixel_count = (pred_np == cls_idx).sum()
        cam_intensity = float(cam.max())
        heatmaps[cls_idx] = (cam, cam_intensity, pixel_count)
        print(f"  [{cls_idx:2d}] {name:15s} | max={cam_intensity:.3f}, pixels={pixel_count}")

    gradcam.remove_hooks()

    # Pick top 4 by Grad-CAM max intensity and save each separately
    ranked = sorted(heatmaps.items(), key=lambda kv: kv[1][1], reverse=True)
    top4 = ranked[:4]

    print(f"\nSaving top 4 Grad-CAM results:")
    for rank, (cls_idx, (cam, intensity, px)) in enumerate(top4, 1):
        name = CLASS_NAMES[cls_idx]
        pixel_pct = px / pred_np.size * 100

        # Overlay: heatmap blended with original
        overlay = overlay_heatmap(rgb_resized, cam, alpha=alpha)
        overlay_path = os.path.join(out_dir, f"{img_stem}_gradcam_top{rank}_{name}_overlay.png")
        cv2.imwrite(overlay_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

        # Raw heatmap: jet colormap only
        heatmap_img = apply_colormap(cam)
        heatmap_path = os.path.join(out_dir, f"{img_stem}_gradcam_top{rank}_{name}_heatmap.png")
        cv2.imwrite(heatmap_path, cv2.cvtColor(heatmap_img, cv2.COLOR_RGB2BGR))

        print(f"  #{rank} {name:15s} | intensity={intensity:.3f}, coverage={pixel_pct:.1f}%")
        print(f"       {overlay_path}")
        print(f"       {heatmap_path}")

    print(f"\nDone. All files saved to: {out_dir}")


# =============================================================================
# RUN — edit these paths for your environment, then run the cell
# =============================================================================
if __name__ == "__main__":
    #   EDIT THESE FOR KAGGLE / LOCAL USE  
    CKPT_PATH  = "/kaggle/working/outputs/best.pt"
    IMAGE_PATH = "/kaggle/input/datasets/jawadulkarim117/cityscapes/Cityscapes_Aug/Cityscapes_Aug/train/images/train_00489.png"
    OUTPUT     = "/kaggle/working/gradcam_output.png"
    #         

    main(
        checkpoint=CKPT_PATH,
        image=IMAGE_PATH,
        output=OUTPUT,
        only_present=True,         # only classes in the prediction
        target_layer_name="decoder.smooth1",  # try: backbone.layer4, fuse3, edge.block3
        alpha=0.5,
        individual=False,          # set True to save each class as a separate file
    )